# LLM Inference Optimization Experimental Pipeline

**Authors:** Waqas Ahmed (i232540), Huzaifa Khalid (i232508)

This notebook runs a full reproducible pipeline across:
- Phase 2 baseline latency benchmarking
- Phase 3 KV prefill evaluation (cold vs warm)
- Phase 4 speculative decoding sweep (two draft models, k = 1/4/8)
- Phase 4 KV eviction comparison (unconstrained, LRU, adaptive)
- Aggregation, visualization, and final evidence-based conclusions


In [ ]:
os.chdir('/pdc_inference')

import os
import sys
import subprocess
import asyncio
import json
import csv
import time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import aiohttp

RESULTS_DIR = Path('results')
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_MODEL_PATH = 'models/mistral-7b-instruct-v0.2.Q8_0.gguf'
DRAFT_MODEL_Q4 = 'models/mistral-7b-instruct-v0.2.Q4_K_M.gguf'
DRAFT_MODEL_Q2 = 'models/mistral-7b-instruct-v0.2.Q2_K.gguf'


def _as_float(x, default=np.nan):
    try:
        if x is None or x == '':
            return default
        return float(x)
    except Exception:
        return default


def _as_int(x, default=0):
    try:
        if x is None or x == '':
            return default
        return int(float(x))
    except Exception:
        return default


def _as_bool(x):
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    return s in {'1', 'true', 'yes', 'y'}


async def _poll_server_async(port, timeout=60):
    url = f'http://localhost:{port}/health'
    deadline = time.time() + timeout
    last_error = ''
    async with aiohttp.ClientSession() as session:
        while time.time() < deadline:
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=5)) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        if data.get('model_loaded'):
                            return True, data
                        last_error = f"model_loaded={data.get('model_loaded')}"
                    else:
                        last_error = f'HTTP {resp.status}'
            except Exception as e:
                last_error = str(e)
            await asyncio.sleep(3)
    print(f'[poll_server] timeout on port {port}: {last_error}')
    return False, {}


def poll_server(port, timeout=60):
    return asyncio.run(_poll_server_async(port, timeout=timeout))


def terminate_server(proc):
    if proc is None:
        return
    try:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=20)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=10)
    finally:
        asyncio.run(asyncio.sleep(3))


def load_csv(path):
    path = Path(path)
    if not path.exists():
        print(f'[warning] missing CSV: {path}')
        return []
    with path.open('r', newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def filter_ok(rows):
    out = []
    for r in rows:
        err = str(r.get('error', '')).strip()
        if err == '':
            out.append(r)
    return out


def pct(data, p):
    vals = [float(x) for x in data if x is not None and not pd.isna(x)]
    if not vals:
        return np.nan
    return float(np.percentile(vals, p))


print('Setup complete.')
print('Working directory:', os.getcwd())
print('Results dir:', RESULTS_DIR.resolve())


## Phase 2: Baseline

In [ ]:
import argparse
import importlib

env = os.environ.copy()
env.update({
    'MODEL_PATH': TARGET_MODEL_PATH,
    'N_GPU_LAYERS': '-1',
    'N_CTX': '2048',
    'N_THREADS': '4',
})

proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

try:
    ok, health = poll_server(8000, timeout=60)
    if not ok:
        raise RuntimeError('Baseline server failed to become ready on port 8000')

    benchmark = importlib.import_module('benchmark')
    args = argparse.Namespace(
        host='http://localhost:8000',
        mode='both',
        n=20,
        concurrency=1,
        out_dir='results',
    )
    asyncio.run(benchmark._main(args))
finally:
    terminate_server(proc)

expected = [
    RESULTS_DIR / 'results_autocomplete_baseline_c1.csv',
    RESULTS_DIR / 'results_rewrite_baseline_c1.csv',
]
for p in expected:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')


In [ ]:
auto_rows = filter_ok(load_csv(RESULTS_DIR / 'results_autocomplete_baseline_c1.csv'))
rew_rows = filter_ok(load_csv(RESULTS_DIR / 'results_rewrite_baseline_c1.csv'))


def baseline_metrics(rows, ttft_target_ms):
    if not rows:
        return {
            'n': 0,
            'ttft_mean': np.nan,
            'ttft_p50': np.nan,
            'ttft_p95': np.nan,
            'tpot_mean': np.nan,
            'compliance_pct': np.nan,
        }
    ttfts = [_as_float(r.get('ttft_ms')) for r in rows]
    tpots = [_as_float(r.get('tpot_ms')) for r in rows if _as_float(r.get('tpot_ms')) > 0]
    compliance = sum(1 for v in ttfts if not pd.isna(v) and v < ttft_target_ms) / max(len(ttfts), 1) * 100.0
    return {
        'n': len(rows),
        'ttft_mean': float(np.nanmean(ttfts)),
        'ttft_p50': pct(ttfts, 50),
        'ttft_p95': pct(ttfts, 95),
        'tpot_mean': float(np.nanmean(tpots)) if tpots else np.nan,
        'compliance_pct': compliance,
    }

baseline_summary_df = pd.DataFrame([
    {'endpoint': 'autocomplete', **baseline_metrics(auto_rows, 200.0)},
    {'endpoint': 'rewrite', **baseline_metrics(rew_rows, 400.0)},
])

if not baseline_summary_df.empty:
    baseline_autocomplete_tpot_p50 = pct([_as_float(r.get('tpot_ms')) for r in auto_rows], 50)
else:
    baseline_autocomplete_tpot_p50 = np.nan

baseline_summary_df


## Phase 3: KV Prefill

In [ ]:
import argparse
import importlib

env = os.environ.copy()
env.update({
    'MODEL_PATH': TARGET_MODEL_PATH,
    'N_GPU_LAYERS': '-1',
    'N_CTX': '4096',
    'N_THREADS': '4',
    'PREFILL_ENABLED': '1',
    'PREFILL_TOKENS': '32',
    'PREFILL_IDLE_MS': '150',
})

proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'prefill_server:app', '--host', '0.0.0.0', '--port', '8001'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

try:
    ok, health = poll_server(8001, timeout=60)
    if not ok:
        raise RuntimeError('Prefill server failed to become ready on port 8001')

    experiment_prefill = importlib.import_module('experiment_prefill')
    args = argparse.Namespace(n=30, workload='autocomplete', out_dir='results')
    asyncio.run(experiment_prefill._main(args))
finally:
    terminate_server(proc)

for p in [RESULTS_DIR / 'phase3_prefill_cold.csv', RESULTS_DIR / 'phase3_prefill_warm.csv']:
    print(f'{p}:', 'OK' if p.exists() else 'MISSING')


In [ ]:
cold_rows = filter_ok(load_csv(RESULTS_DIR / 'phase3_prefill_cold.csv'))
warm_rows = filter_ok(load_csv(RESULTS_DIR / 'phase3_prefill_warm.csv'))


def prefill_stats(rows):
    if not rows:
        return {
            'n': 0,
            'ttft_mean': np.nan,
            'ttft_p50': np.nan,
            'ttft_p95': np.nan,
            'cache_hit_rate': np.nan,
        }
    ttfts = [_as_float(r.get('ttft_ms')) for r in rows]
    hit_rate = np.nan
    if 'cache_hit' in rows[0]:
        hit_rate = float(np.mean([1.0 if _as_bool(r.get('cache_hit')) else 0.0 for r in rows]))
    return {
        'n': len(rows),
        'ttft_mean': float(np.nanmean(ttfts)),
        'ttft_p50': pct(ttfts, 50),
        'ttft_p95': pct(ttfts, 95),
        'cache_hit_rate': hit_rate,
    }

cold_m = prefill_stats(cold_rows)
warm_m = prefill_stats(warm_rows)

prefill_stats_df = pd.DataFrame([
    {'condition': 'cold', **cold_m},
    {'condition': 'warm', **warm_m},
])

if cold_m['n'] > 0 and warm_m['n'] > 0:
    prefill_comparison_df = pd.DataFrame([
        {
            'metric': 'TTFT mean (ms)',
            'cold': cold_m['ttft_mean'],
            'warm': warm_m['ttft_mean'],
            'reduction_pct': (cold_m['ttft_mean'] - warm_m['ttft_mean']) / max(cold_m['ttft_mean'], 1e-9) * 100.0,
        },
        {
            'metric': 'TTFT p50 (ms)',
            'cold': cold_m['ttft_p50'],
            'warm': warm_m['ttft_p50'],
            'reduction_pct': (cold_m['ttft_p50'] - warm_m['ttft_p50']) / max(cold_m['ttft_p50'], 1e-9) * 100.0,
        },
        {
            'metric': 'TTFT p95 (ms)',
            'cold': cold_m['ttft_p95'],
            'warm': warm_m['ttft_p95'],
            'reduction_pct': (cold_m['ttft_p95'] - warm_m['ttft_p95']) / max(cold_m['ttft_p95'], 1e-9) * 100.0,
        },
        {
            'metric': 'Cache hit rate',
            'cold': np.nan,
            'warm': warm_m['cache_hit_rate'],
            'reduction_pct': np.nan,
        },
    ])
else:
    prefill_comparison_df = pd.DataFrame()

prefill_comparison_df


## Phase 4: Speculative Decoding

In [ ]:
import argparse
import importlib

experiment_speculative = importlib.import_module('experiment_speculative')

spec_rows_q4 = []
for k in [1, 4, 8]:
    env = os.environ.copy()
    env.update({
        'MODEL_PATH': TARGET_MODEL_PATH,
        'DRAFT_MODEL_PATH': DRAFT_MODEL_Q4,
        'N_GPU_LAYERS': '-1',
        'N_CTX': '2048',
        'N_THREADS': '4',
        'SPEC_ENABLED': '1',
        'SPEC_K': str(k),
        'SPEC_BASELINE_TPOT_MS': str(_as_float(globals().get('baseline_autocomplete_tpot_p50'), 0.0) or 0.0),
        'EVICTION_POLICY': 'lru',
        'EVICTION_MAX_CTX': '1024',
    })

    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'phase4_server:app', '--host', '0.0.0.0', '--port', '8002'],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    try:
        ok, health = poll_server(8002, timeout=60)
        if not ok:
            raise RuntimeError(f'Phase4 server not ready for Q4 draft k={k}')
        if not health.get('spec_active', False):
            raise RuntimeError(f"Speculative mode inactive for Q4 draft k={k}: {health.get('spec_disabled_reason', 'unknown reason')}")

        args = argparse.Namespace(n=20, k_values=str(k), eviction='lru', out_dir='results', manual=True)
        asyncio.run(experiment_speculative._main(args))

        rows = filter_ok(load_csv(RESULTS_DIR / 'phase4_speculative_results.csv'))
        for r in rows:
            r['draft_model'] = 'Q4_K_M'
            r['k'] = _as_int(r.get('k', k), k)
        spec_rows_q4.extend(rows)

        pd.DataFrame(rows).to_csv(RESULTS_DIR / f'phase4_speculative_results_Q4_K_M_k{k}.csv', index=False)
    finally:
        terminate_server(proc)

print('Collected Q4 rows:', len(spec_rows_q4))


In [ ]:
import argparse
import importlib

experiment_speculative = importlib.import_module('experiment_speculative')

spec_rows_q2 = []
for k in [1, 4, 8]:
    env = os.environ.copy()
    env.update({
        'MODEL_PATH': TARGET_MODEL_PATH,
        'DRAFT_MODEL_PATH': DRAFT_MODEL_Q2,
        'N_GPU_LAYERS': '-1',
        'N_CTX': '2048',
        'N_THREADS': '4',
        'SPEC_ENABLED': '1',
        'SPEC_K': str(k),
        'SPEC_BASELINE_TPOT_MS': str(_as_float(globals().get('baseline_autocomplete_tpot_p50'), 0.0) or 0.0),
        'EVICTION_POLICY': 'lru',
        'EVICTION_MAX_CTX': '1024',
    })

    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'phase4_server:app', '--host', '0.0.0.0', '--port', '8002'],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    try:
        ok, health = poll_server(8002, timeout=60)
        if not ok:
            raise RuntimeError(f'Phase4 server not ready for Q2 draft k={k}')
        if not health.get('spec_active', False):
            raise RuntimeError(f"Speculative mode inactive for Q2 draft k={k}: {health.get('spec_disabled_reason', 'unknown reason')}")

        args = argparse.Namespace(n=20, k_values=str(k), eviction='lru', out_dir='results', manual=True)
        asyncio.run(experiment_speculative._main(args))

        rows = filter_ok(load_csv(RESULTS_DIR / 'phase4_speculative_results.csv'))
        for r in rows:
            r['draft_model'] = 'Q2_K'
            r['k'] = _as_int(r.get('k', k), k)
        spec_rows_q2.extend(rows)

        pd.DataFrame(rows).to_csv(RESULTS_DIR / f'phase4_speculative_results_Q2_K_k{k}.csv', index=False)
    finally:
        terminate_server(proc)

print('Collected Q2 rows:', len(spec_rows_q2))


In [ ]:
spec_rows_all = []
spec_rows_all.extend(globals().get('spec_rows_q4', []))
spec_rows_all.extend(globals().get('spec_rows_q2', []))

spec_df = pd.DataFrame(spec_rows_all)
if spec_df.empty:
    print('[warning] No speculative results collected.')
    spec_summary_df = pd.DataFrame(columns=['draft_model', 'k', 'TTFT_p50', 'TPOT_p50', 'acceptance_rate', 'speedup_ratio'])
else:
    for c in ['ttft_ms', 'tpot_ms', 'acceptance_rate', 'speedup_ratio']:
        spec_df[c] = pd.to_numeric(spec_df[c], errors='coerce')
    spec_df['k'] = pd.to_numeric(spec_df['k'], errors='coerce').astype('Int64')

    rows = []
    for (draft, k), g in spec_df.groupby(['draft_model', 'k'], dropna=True):
        rows.append({
            'draft_model': draft,
            'k': int(k),
            'TTFT_p50': pct(g['ttft_ms'].tolist(), 50),
            'TPOT_p50': pct(g['tpot_ms'].tolist(), 50),
            'acceptance_rate': float(np.nanmean(g['acceptance_rate'])),
            'speedup_ratio': float(np.nanmean(g['speedup_ratio'])),
        })
    spec_summary_df = pd.DataFrame(rows).sort_values(['draft_model', 'k']).reset_index(drop=True)

spec_summary_df


## Phase 4: Eviction Experiment

In [ ]:
from workload_gen import generate_revision_history_workload

revision_workload = generate_revision_history_workload(n=20)


async def _send_eviction_request(session, prompt, condition):
    row = {
        'condition': condition,
        'prompt_len': len(prompt),
        'ttft_ms': np.nan,
        'e2e_ms': np.nan,
        'response_len': 0,
        'vram_used_mb': np.nan,
        'error': '',
    }
    payload = {'prompt': prompt, 'max_tokens': 64, 'temperature': 0.1}
    try:
        async with session.post('http://localhost:8002/complete_spec', json=payload, timeout=aiohttp.ClientTimeout(total=120)) as resp:
            if resp.status != 200:
                row['error'] = f'HTTP {resp.status}: {(await resp.text())[:120]}'
                return row
            data = await resp.json()
            e2e = _as_float(data.get('e2e_ms'), np.nan)
            ttft = _as_float(data.get('ttft_ms'), e2e)
            row['e2e_ms'] = e2e
            row['ttft_ms'] = ttft
            row['response_len'] = len(data.get('text', ''))
            mem = data.get('mem', {}) if isinstance(data, dict) else {}
            vram = mem.get('vram_used_mb', np.nan) if isinstance(mem, dict) else np.nan
            if pd.isna(vram):
                try:
                    async with session.get('http://localhost:8002/health', timeout=aiohttp.ClientTimeout(total=5)) as hr:
                        h = await hr.json()
                        vram = _as_float((h.get('memory') or {}).get('vram_used_mb'), np.nan)
                except Exception:
                    pass
            row['vram_used_mb'] = vram
    except Exception as e:
        row['error'] = f'{type(e).__name__}: {e}'
    return row


async def _run_eviction_condition_async(condition, workload, warmup=2):
    rows = []
    connector = aiohttp.TCPConnector(limit=2)
    async with aiohttp.ClientSession(connector=connector) as session:
        for p in workload[:warmup]:
            await _send_eviction_request(session, p['prompt'], condition)
        for p in workload[warmup:]:
            rows.append(await _send_eviction_request(session, p['prompt'], condition))
    return rows


conditions = [
    ('unconstrained', 'lru', 2048),
    ('lru', 'lru', 256),
    ('adaptive', 'adaptive', 256),
]

all_eviction_rows = []
for condition, policy, max_ctx in conditions:
    env = os.environ.copy()
    env.update({
        'MODEL_PATH': TARGET_MODEL_PATH,
        'N_GPU_LAYERS': '-1',
        'N_CTX': '2048',
        'N_THREADS': '4',
        'SPEC_ENABLED': '0',
        'EVICTION_POLICY': policy,
        'EVICTION_MAX_CTX': str(max_ctx),
    })

    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'phase4_server:app', '--host', '0.0.0.0', '--port', '8002'],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    try:
        ok, _ = poll_server(8002, timeout=60)
        if not ok:
            print(f'[warning] skipping condition={condition}; server failed to become ready')
            continue
        rows = asyncio.run(_run_eviction_condition_async(condition, revision_workload, warmup=2))
        all_eviction_rows.extend(rows)
    finally:
        terminate_server(proc)

# Persist exactly the expected Phase 4 eviction CSV schema
phase4_eviction_csv = RESULTS_DIR / 'phase4_eviction_results.csv'
with phase4_eviction_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['condition', 'prompt_len', 'ttft_ms', 'e2e_ms', 'response_len', 'vram_used_mb', 'error'],
    )
    writer.writeheader()
    for r in all_eviction_rows:
        writer.writerow(r)

print('Saved:', phase4_eviction_csv)
print('Rows:', len(all_eviction_rows))


In [ ]:
ev_rows = filter_ok(load_csv(RESULTS_DIR / 'phase4_eviction_results.csv'))
ev_df = pd.DataFrame(ev_rows)

if ev_df.empty:
    print('[warning] No eviction results available.')
    eviction_summary_df = pd.DataFrame(columns=['condition', 'TTFT_p50', 'TTFT_p95', 'VRAM_avg', 'response_ratio'])
else:
    for c in ['prompt_len', 'ttft_ms', 'response_len', 'vram_used_mb']:
        ev_df[c] = pd.to_numeric(ev_df[c], errors='coerce')

    baseline_resp = np.nan
    if (ev_df['condition'] == 'unconstrained').any():
        baseline_resp = ev_df.loc[ev_df['condition'] == 'unconstrained', 'response_len'].mean()

    rows = []
    for cond, g in ev_df.groupby('condition'):
        avg_resp = g['response_len'].mean()
        response_ratio = np.nan
        if not pd.isna(baseline_resp) and baseline_resp > 0:
            response_ratio = avg_resp / baseline_resp
        rows.append({
            'condition': cond,
            'TTFT_p50': pct(g['ttft_ms'].tolist(), 50),
            'TTFT_p95': pct(g['ttft_ms'].tolist(), 95),
            'VRAM_avg': float(np.nanmean(g['vram_used_mb'])),
            'response_ratio': response_ratio,
        })

    eviction_summary_df = pd.DataFrame(rows).sort_values('condition').reset_index(drop=True)

eviction_summary_df


In [ ]:
master_rows = []

# Phase 2 baseline
if 'baseline_summary_df' in globals() and not baseline_summary_df.empty:
    for _, r in baseline_summary_df.iterrows():
        master_rows.append({
            'phase': 'Phase 2',
            'experiment': f"baseline_{r['endpoint']}",
            'draft_model': np.nan,
            'k': np.nan,
            'eviction_policy': np.nan,
            'ttft_p50': r['ttft_p50'],
            'ttft_p95': r['ttft_p95'],
            'tpot_p50': np.nan,
            'acceptance_rate': np.nan,
            'speedup_ratio': np.nan,
            'vram_mb': np.nan,
        })

# Phase 3 prefill
if 'prefill_stats_df' in globals() and not prefill_stats_df.empty:
    for _, r in prefill_stats_df.iterrows():
        master_rows.append({
            'phase': 'Phase 3',
            'experiment': f"prefill_{r['condition']}",
            'draft_model': np.nan,
            'k': np.nan,
            'eviction_policy': np.nan,
            'ttft_p50': r['ttft_p50'],
            'ttft_p95': r['ttft_p95'],
            'tpot_p50': np.nan,
            'acceptance_rate': r.get('cache_hit_rate', np.nan) if r['condition'] == 'warm' else np.nan,
            'speedup_ratio': np.nan,
            'vram_mb': np.nan,
        })

# Phase 4 speculative
if 'spec_summary_df' in globals() and not spec_summary_df.empty:
    for _, r in spec_summary_df.iterrows():
        master_rows.append({
            'phase': 'Phase 4',
            'experiment': 'speculative',
            'draft_model': r['draft_model'],
            'k': r['k'],
            'eviction_policy': 'lru',
            'ttft_p50': r['TTFT_p50'],
            'ttft_p95': np.nan,
            'tpot_p50': r['TPOT_p50'],
            'acceptance_rate': r['acceptance_rate'],
            'speedup_ratio': r['speedup_ratio'],
            'vram_mb': np.nan,
        })

# Phase 4 eviction
if 'eviction_summary_df' in globals() and not eviction_summary_df.empty:
    for _, r in eviction_summary_df.iterrows():
        master_rows.append({
            'phase': 'Phase 4',
            'experiment': 'eviction',
            'draft_model': np.nan,
            'k': np.nan,
            'eviction_policy': r['condition'],
            'ttft_p50': r['TTFT_p50'],
            'ttft_p95': r['TTFT_p95'],
            'tpot_p50': np.nan,
            'acceptance_rate': np.nan,
            'speedup_ratio': np.nan,
            'vram_mb': r['VRAM_avg'],
        })

all_results_df = pd.DataFrame(master_rows, columns=[
    'phase', 'experiment', 'draft_model', 'k', 'eviction_policy',
    'ttft_p50', 'ttft_p95', 'tpot_p50', 'acceptance_rate', 'speedup_ratio', 'vram_mb'
])

all_results_csv = RESULTS_DIR / 'all_results.csv'
all_results_json = RESULTS_DIR / 'all_results.json'
all_results_df.to_csv(all_results_csv, index=False)
all_results_df.to_json(all_results_json, orient='records', indent=2)

print('Saved:', all_results_csv)
print('Saved:', all_results_json)
all_results_df


## Visualization

In [ ]:
import importlib

if not Path('plot_results.py').exists():
    print('[warning] plot_results.py not found; skipping baseline/prefill plots.')
else:
    plot_results = importlib.import_module('plot_results')
    importlib.reload(plot_results)

    if hasattr(plot_results, 'fig1_ttft_baseline'):
        plot_results.fig1_ttft_baseline()
        plt.show()
    else:
        print('[warning] fig1_ttft_baseline() not found in plot_results.py')

    if hasattr(plot_results, 'fig3_prefill_comparison'):
        plot_results.fig3_prefill_comparison()
        plt.show()
    else:
        print('[warning] fig3_prefill_comparison() not found in plot_results.py')


In [ ]:
import importlib

if not Path('plot_results.py').exists():
    print('[warning] plot_results.py not found; skipping speculative plots.')
else:
    plot_results = importlib.import_module('plot_results')
    importlib.reload(plot_results)

    if hasattr(plot_results, 'fig4_spec_acceptance'):
        plot_results.fig4_spec_acceptance()
        if 'spec_summary_df' in globals() and not spec_summary_df.empty:
            q2 = spec_summary_df[spec_summary_df['draft_model'] == 'Q2_K'].sort_values('k')
            if not q2.empty:
                ax = plt.gca()
                ax.plot(q2['k'], q2['acceptance_rate'], marker='o', linestyle='--', label='Q2_K overlay')
                ax.legend()
        plt.show()
    else:
        print('[warning] fig4_spec_acceptance() not found in plot_results.py')

    if hasattr(plot_results, 'fig5_spec_tpot'):
        plot_results.fig5_spec_tpot()
        if 'spec_summary_df' in globals() and not spec_summary_df.empty:
            q2 = spec_summary_df[spec_summary_df['draft_model'] == 'Q2_K'].sort_values('k')
            if not q2.empty:
                ax = plt.gca()
                ax.plot(q2['k'], q2['TPOT_p50'], marker='o', linestyle='--', label='Q2_K overlay')
                ax.legend()
        plt.show()
    else:
        print('[warning] fig5_spec_tpot() not found in plot_results.py')


In [ ]:
import importlib

if not Path('plot_results.py').exists():
    print('[warning] plot_results.py not found; skipping eviction plots.')
else:
    plot_results = importlib.import_module('plot_results')
    importlib.reload(plot_results)

    if hasattr(plot_results, 'fig6_eviction_ttft'):
        plot_results.fig6_eviction_ttft()
        plt.show()
    else:
        print('[warning] fig6_eviction_ttft() not found in plot_results.py')

    if hasattr(plot_results, 'fig7_eviction_quality'):
        plot_results.fig7_eviction_quality()
        plt.show()
    else:
        print('[warning] fig7_eviction_quality() not found in plot_results.py')


In [ ]:
import importlib

if not Path('plot_results.py').exists():
    print('[warning] plot_results.py not found; skipping summary dashboard plot.')
else:
    plot_results = importlib.import_module('plot_results')
    importlib.reload(plot_results)

    if hasattr(plot_results, 'fig9_summary_dashboard'):
        plot_results.fig9_summary_dashboard()
        plt.show()
    else:
        print('[warning] fig9_summary_dashboard() not found in plot_results.py')


In [ ]:
print('Final Analysis')
print('=' * 80)

# Baseline performance
if 'baseline_summary_df' in globals() and not baseline_summary_df.empty:
    ac = baseline_summary_df[baseline_summary_df['endpoint'] == 'autocomplete']
    rw = baseline_summary_df[baseline_summary_df['endpoint'] == 'rewrite']

    if not ac.empty:
        ac_mean = float(ac['ttft_mean'].iloc[0])
        ac_p50 = float(ac['ttft_p50'].iloc[0])
        delta_200 = ac_p50 - 200.0
        print(f"Baseline performance (autocomplete): mean TTFT={ac_mean:.1f} ms, p50={ac_p50:.1f} ms, gap vs 200 ms target={delta_200:.1f} ms")

    if not rw.empty:
        rw_mean = float(rw['ttft_mean'].iloc[0])
        rw_p50 = float(rw['ttft_p50'].iloc[0])
        delta_200_rw = rw_p50 - 200.0
        print(f"Baseline performance (rewrite): mean TTFT={rw_mean:.1f} ms, p50={rw_p50:.1f} ms, gap vs 200 ms target={delta_200_rw:.1f} ms")
else:
    print('Baseline performance: unavailable (missing baseline CSVs).')

# Prefill effectiveness
if 'prefill_comparison_df' in globals() and not prefill_comparison_df.empty:
    m = prefill_comparison_df.set_index('metric')
    mean_red = float(m.loc['TTFT mean (ms)', 'reduction_pct'])
    p95_red = float(m.loc['TTFT p95 (ms)', 'reduction_pct'])
    hit_rate = float(m.loc['Cache hit rate', 'warm']) if not pd.isna(m.loc['Cache hit rate', 'warm']) else np.nan
    cold_mean = float(m.loc['TTFT mean (ms)', 'cold'])
    warm_mean = float(m.loc['TTFT mean (ms)', 'warm'])
    help_text = 'helps' if warm_mean < cold_mean else 'hurts'
    print(f"Prefill effectiveness: mean TTFT reduction={mean_red:.1f}%, p95 reduction={p95_red:.1f}%, cache hit rate={hit_rate:.1%}; observed net effect {help_text} on average TTFT.")
else:
    print('Prefill effectiveness: unavailable (missing prefill CSVs).')

# Eviction impact
if 'eviction_summary_df' in globals() and not eviction_summary_df.empty:
    ev = eviction_summary_df.set_index('condition')
    if {'lru', 'adaptive'}.issubset(set(ev.index)):
        lru_ttft = float(ev.loc['lru', 'TTFT_p50'])
        ad_ttft = float(ev.loc['adaptive', 'TTFT_p50'])
        lru_q = float(ev.loc['lru', 'response_ratio']) if not pd.isna(ev.loc['lru', 'response_ratio']) else np.nan
        ad_q = float(ev.loc['adaptive', 'response_ratio']) if not pd.isna(ev.loc['adaptive', 'response_ratio']) else np.nan
        print(f"Eviction impact: LRU p50 TTFT={lru_ttft:.1f} ms, Adaptive p50 TTFT={ad_ttft:.1f} ms; response ratio LRU={lru_q:.3f}, Adaptive={ad_q:.3f} (higher preserves quality proxy better).")
    else:
        print('Eviction impact: partial condition coverage only.')
else:
    print('Eviction impact: unavailable (missing eviction CSV).')

# Speculative tradeoffs and Q4 vs Q2
if 'spec_summary_df' in globals() and not spec_summary_df.empty:
    sp = spec_summary_df.copy()
    best_speed = sp.loc[sp['speedup_ratio'].idxmax()]
    best_tpot = sp.loc[sp['TPOT_p50'].idxmin()]
    avg_accept = float(sp['acceptance_rate'].mean())
    print(f"Speculative tradeoffs: average acceptance={avg_accept:.3f}; best measured speedup={best_speed['speedup_ratio']:.3f}x at {best_speed['draft_model']} k={int(best_speed['k'])}; lowest TPOT p50={best_tpot['TPOT_p50']:.2f} ms/tok at {best_tpot['draft_model']} k={int(best_tpot['k'])}.")

    q4 = sp[sp['draft_model'] == 'Q4_K_M']
    q2 = sp[sp['draft_model'] == 'Q2_K']
    if not q4.empty and not q2.empty:
        q4_best = q4.loc[q4['speedup_ratio'].idxmax()]
        q2_best = q2.loc[q2['speedup_ratio'].idxmax()]
        better = 'Q4_K_M' if q4_best['speedup_ratio'] >= q2_best['speedup_ratio'] else 'Q2_K'
        print(f"Q4 vs Q2 draft comparison: best Q4 speedup={q4_best['speedup_ratio']:.3f}x, best Q2 speedup={q2_best['speedup_ratio']:.3f}x; better draft in this run: {better}.")
    else:
        print('Q4 vs Q2 draft comparison: incomplete data for one draft model.')
else:
    print('Speculative decoding tradeoffs: unavailable (missing speculative results).')

print('Why real speedup is below theoretical: both draft and target passes are still serialized on a single GPU; kernel launch overhead, memory bandwidth pressure, and verification cost cap practical gains.')
print('Combined picture: prefill primarily targets TTFT reduction, speculative decoding targets token throughput/TPOT, and eviction preserves latency under context pressure by controlling KV growth.')
